# S1 -> S2 FILE TRANSFER

This script removes the unnesessary files and tranfers files to be used in this project to the corresponding folders

Objective: remove as few files as possible, but leave enough space on the hard drive for suture analysis

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.paths import PROJECT_1_EEG_DATA_DIR
import config.config as cfg

import os
import shutil


REMOVING FILES

In [ ]:
# Step 1: remove folders with figures and resting state recordings

for group_name in cfg.GROUPS:

    group_dir = PROJECT_1_EEG_DATA_DIR / group_name

    sub_names = os.listdir(group_dir)

    for sub_name in sub_names:

        pac_dir = os.path.join(PROJECT_1_EEG_DATA_DIR,
            group_name,
            sub_name,
            "pac_results"
        )

        rest_dir = os.path.join(PROJECT_1_EEG_DATA_DIR,
            group_name,
            sub_name,
            "rest"
        )

        if os.path.exists(pac_dir):

            print(f"Removing {pac_dir}")
            shutil.rmtree(pac_dir)
        
        if os.path.exists(rest_dir):

            print(f"Removing {rest_dir}")
            shutil.rmtree(rest_dir)

In [ ]:
# Step 2: remove raw eeg recordings

for group_name in cfg.GROUPS:

    group_dir = PROJECT_1_EEG_DATA_DIR / group_name

    sub_dirs = [p for p in group_dir.iterdir() if p.is_dir()]

    for sub_dir in sub_dirs:

        files = [p for p in sub_dir.iterdir() if p.is_file()]

        for file in files:

            print(f"Removing {file}")

            file.unlink()

print("Done.")

In [2]:
# Step 3: Remove ICA folders
for group_name in cfg.GROUPS:

    group_dir = PROJECT_1_EEG_DATA_DIR / group_name

    sub_dirs = [p for p in group_dir.iterdir() if p.is_dir()]

    for sub_dir in sub_dirs:

        ica_dir = sub_dir / "preproc" / "ICA"
        # print(ica_dir)

        if ica_dir.exists():
            print(f"Removing {ica_dir}")
            shutil.rmtree(ica_dir)

In [ ]:
# Step 4: Remove no bads files

for group_name in cfg.GROUPS:

    group_dir = PROJECT_1_EEG_DATA_DIR / group_name

    sub_dirs = [p for p in group_dir.iterdir() if p.is_dir()]

    for sub_dir in sub_dirs:

        preproc_dir = sub_dir / "preproc"

        if preproc_dir.exists():

            for file in preproc_dir.iterdir():

                if file.is_file() and "_nobads" in file.name:

                    print(f"Removing: {file}")
                    file.unlink()

In [ ]:
# Step 5: Remove TFR files

for group_name in cfg.GROUPS:

    group_dir = PROJECT_1_EEG_DATA_DIR / group_name

    sub_dirs = [p for p in group_dir.iterdir() if p.is_dir()]

    for sub_dir in sub_dirs:

        preproc_dir = sub_dir / "preproc"
        analysis_dir = preproc_dir / "analysis"

        if preproc_dir.exists() and analysis_dir.exists():

            for file in analysis_dir.iterdir():

                if file.is_file() and "-tfr" in file.name:

                    print(f"Removing: {file}")
                    file.unlink()

In [ ]:
# Step 6: Remove duplicated and band-filtered morphed source files

bands = {"theta", "alpha", "beta", "gamma"}

for group_name in cfg.GROUPS:

    group_dir = PROJECT_1_EEG_DATA_DIR / group_name

    sub_dirs = [p for p in group_dir.iterdir() if p.is_dir()]

    for sub_dir in sub_dirs:

        source_dir = sub_dir / "preproc" / "analysis" / "source"
        morphed_source_dir = source_dir / "morphed_stcs"

        if source_dir.exists():
            stcs_dir = source_dir / "stcs"
            for file in stcs_dir.iterdir():
                if file.is_file() and file.suffix == ".stc":
                    print(f"Removing: {file}")
                    file.unlink()

        if morphed_source_dir.exists():
            # Search recursively through all subdirectories
            for folder in morphed_source_dir.rglob("*"):

                if folder.is_dir() and folder.name in bands:

                    print(f"Removing: {folder}")

                    shutil.rmtree(folder)

TRANSFERRING FILES

In [ ]:
# Step 1: Create new folder structure for sensor and source epochs

from pathlib import Path
import config.config as cfg
from config.paths import PROJECT_1_EEG_DATA_DIR, STUDY_2_EPOCHS_DIR


# New folders
LEVELS = {
    "sensor": STUDY_2_EPOCHS_DIR / "sensor",
    "source": STUDY_2_EPOCHS_DIR / "source",
}

TASK = "FTT"

for group_name in cfg.GROUPS:

    # Old project group folder
    old_group_dir = PROJECT_1_EEG_DATA_DIR / group_name

    if not old_group_dir.exists():
        print(f"Missing: {old_group_dir}")
        continue

    # Find subjects
    subjects = [
        p.name for p in old_group_dir.iterdir()
        if p.is_dir()
    ]

    print(f"{group_name}: found {len(subjects)} subjects")

    for subject in subjects:

        for level, level_dir in LEVELS.items():

            subject_dir = (
                level_dir
                / group_name
                / subject
                / TASK
            )

            # Create task stage folders
            for stage in cfg.CONDITIONS:

                stage_dir = subject_dir / stage

                stage_dir.mkdir(
                    parents=True,
                    exist_ok=True
                )

        print(f"Created folders for {group_name}/{subject}")


print("Done.")

In [ ]:
# Step 1: Transfer epochs on sensor and source space to the new project directory

from pathlib import Path
import shutil

from config.paths import (
    PROJECT_1_EEG_DATA_DIR,
    STUDY_2_EPOCHS_DIR
)
import config.config as cfg


TASK = "FTT"

STAGES_MAP = {
    "_plan": "plan",
    "_go": "go",
}


for group in cfg.GROUPS:

    old_group_dir = PROJECT_1_EEG_DATA_DIR / group

    subjects = [
        p for p in old_group_dir.iterdir()
        if p.is_dir()
    ]

    for sub_dir in subjects:

        subject = sub_dir.name

        analysis_dir = sub_dir / "preproc" / "analysis"

        print(f"\nProcessing {group}/{subject}")

        EXCLUDED_NAMES = [
            "MAIN",
        ]

        # ======================================================
        # SENSOR EPOCHS
        # ======================================================

        for epoch_file in analysis_dir.glob("*-epo.fif"):

            # Skip unwanted MAIN epochs
            if any(excluded in epoch_file.name for excluded in EXCLUDED_NAMES):
                print(f"Skipping: {epoch_file.name}")
                continue


            if "_plan" in epoch_file.name:
                stage = "plan"

            elif "_go" in epoch_file.name:
                stage = "GO"

            else:
                continue


            dst = (
                STUDY_2_EPOCHS_DIR
                / "sensor"
                / group
                / subject
                / TASK
                / stage
            )

            dst.mkdir(
                parents=True,
                exist_ok=True
            )

            new_name = epoch_file.name.replace("BL", TASK)

            print(
                f"Copy sensor:\n"
                f"  {epoch_file}\n"
                f"  -> {dst / new_name}"
            )

            shutil.copy2(
                epoch_file,
                dst / new_name
            )

        # ======================================================
        # SOURCE EPOCHS
        # ======================================================

        morphed_root = (
            analysis_dir
            / "source"
            / "morphed_stcs"
            / "_BL"
        )

        if not morphed_root.exists():
            print("No source data")
            continue


        for old_stage, new_stage in STAGES_MAP.items():

            stage_dir = morphed_root / old_stage

            if not stage_dir.exists():
                continue


            dst = (
                STUDY_2_EPOCHS_DIR
                / "source"
                / group
                / subject
                / TASK
                / new_stage
            )

            dst.mkdir(
                parents=True,
                exist_ok=True
            )


            for stc in stage_dir.glob("*.stc"):

                # Skip unwanted MAIN epochs
                if any(excluded in stc.name for excluded in EXCLUDED_NAMES):
                    print(f"Skipping: {stc.name}")
                    continue

                new_name = stc.name.replace("BL", TASK)

                print(
                    f"Copy source:\n"
                    f"  {stc}\n"
                    f"  -> {dst / new_name}"
                )

                shutil.copy2(
                    stc,
                    dst / new_name
                )


print("\nTransfer complete.")


In [3]:
# Step 1: Transfer epochs on sensor and source space to the new project directory

from pathlib import Path
import shutil

from config.paths import (
    PROJECT_1_EEG_DATA_DIR,
    STUDY_2_EPOCHS_DIR
)
import config.config as cfg


TASK = "FTT"

STAGES_MAP = {
    "_plan": "plan",
    "_go": "go",
}


for group in ["Y"]:

    old_group_dir = PROJECT_1_EEG_DATA_DIR / group

    subjects = [
        p for p in old_group_dir.iterdir()
        if p.is_dir()
    ]

    for sub_dir in subjects:

        subject = sub_dir.name
        if subject in ["s1_pac_sub01", "s1_pac_sub07", "s1_pac_sub10", "s1_pac_sub11", "s1_pac_sub22", "s1_pac_sub24"]:
            analysis_dir = sub_dir / "preproc" / "analysis"

            print(f"\nProcessing {group}/{subject}")

            EXCLUDED_NAMES = [
                "MAIN",
            ]

            # ======================================================
            # SOURCE EPOCHS
            # ======================================================

            morphed_root = (
                analysis_dir
                / "source"
                / "morphed_stcs"
                / "_BL"
            )

            if not morphed_root.exists():
                print("No source data")
                continue


            for old_stage, new_stage in STAGES_MAP.items():

                stage_dir = morphed_root / old_stage

                if not stage_dir.exists():
                    continue


                dst = (
                    STUDY_2_EPOCHS_DIR
                    / "source"
                    / group
                    / subject
                    / TASK
                    / new_stage
                )

                dst.mkdir(
                    parents=True,
                    exist_ok=True
                )


                for stc in stage_dir.glob("*.stc"):

                    # Skip unwanted MAIN epochs
                    if any(excluded in stc.name for excluded in EXCLUDED_NAMES):
                        print(f"Skipping: {stc.name}")
                        continue

                    new_name = stc.name.replace("BL", TASK)

                    print(
                        f"Copy source:\n"
                        f"  {stc}\n"
                        f"  -> {dst / new_name}"
                    )

                    shutil.copy2(
                        stc,
                        dst / new_name
                    )


    print("\nTransfer complete.")



Processing Y/s1_pac_sub01
Copy source:
  D:\BonoKat\research project\# study 1\eeg_data\set\Y\s1_pac_sub01\preproc\analysis\source\morphed_stcs\_BL\_plan\s1_pac_sub01_BL_plan-stc-lcmv_epoch_000_morphed-lh.stc
  -> F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\FTT\plan\s1_pac_sub01_FTT_plan-stc-lcmv_epoch_000_morphed-lh.stc
Copy source:
  D:\BonoKat\research project\# study 1\eeg_data\set\Y\s1_pac_sub01\preproc\analysis\source\morphed_stcs\_BL\_plan\s1_pac_sub01_BL_plan-stc-lcmv_epoch_000_morphed-rh.stc
  -> F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\FTT\plan\s1_pac_sub01_FTT_plan-stc-lcmv_epoch_000_morphed-rh.stc
Copy source:
  D:\BonoKat\research project\# study 1\eeg_data\set\Y\s1_pac_sub01\preproc\analysis\source\morphed_stcs\_BL\_plan\s1_pac_sub01_BL_plan-stc-lcmv_epoch_001_morphed-lh.stc
  -> F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\FTT\plan\s1_pac_sub01_FTT_plan-stc-lcmv_epoch_001_morphed-lh.stc
Copy source:
  D:\BonoKat\research project\# study 1\eeg_data